# 04 - Model-based risk analysis: from system model to fault tree

## Learning objectives

- Distinguish system models from risk models
- Understand the legacy YAML exchange model
- Validate identifiers and references
- Trace deterministic model-to-FTA transformations
- Import a native SysML v2 reference-tool export


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


## 1. Why model-based analysis?

A manually maintained fault tree can become inconsistent when the system changes. Establish an explicit connection between the **system description** and the **risk model**.

First study the legacy, manually authored YAML/JSON exchange representation: components, functions, failure modes and hazard logic. It is not itself a native SysML export. The last section then uses the new adapter on a genuine reference-tool export of the supplied `.sysml` model.


## 2. A minimal exchange model

LSHH-101 initiates shutdown and XV-101 stops inflow. Each has a failure-on-demand event. The explicit hazard rule says that either failure defeats automatic shutdown.


In [ ]:
model={
 "components":[
  {"id":"LSHH101","type":"high_high_switch","function":"initiate_shutdown","failure_modes":[{"id":"LSHH101_FOD","mode":"fail_on_demand","probability":.01}]},
  {"id":"XV101","type":"shutdown_valve","function":"stop_inflow","failure_modes":[{"id":"XV101_FOD","mode":"fail_on_demand","probability":.005}]}
 ],
 "hazards":[{"id":"AUTO_FAIL","logic":{"gate":"OR","inputs":[{"event":"LSHH101_FOD"},{"event":"XV101_FOD"}]}}]
}
model


## 3. YAML is an exchange format

The engineering content is in the field definitions, validation rules and transformation, not the YAML syntax. A native-source adapter can populate this same intermediate schema, allowing reuse of the existing FTA functions.


In [ ]:
import yaml
print(yaml.safe_dump(model,sort_keys=False,allow_unicode=True))


## 4. Validate before generating

At minimum, check unique IDs, hazard logic, event references and supported gates. The short teaching validator below is deliberately simpler than the package validator.


In [ ]:
def catalog(m):
    return {fm["id"]:{**fm,"component":c["id"]} for c in m.get("components",[]) for fm in c.get("failure_modes",[])}

def validate(m):
    errors=[]; ids=[c.get("id") for c in m.get("components",[])]
    if len(ids)!=len(set(ids)): errors.append("duplicate component id")
    cat=catalog(m)
    def check(x):
        if "event" in x:
            if x["event"] not in cat: errors.append("unknown event: "+x["event"])
            return
        if str(x.get("gate","")).upper() not in {"AND","OR"}: errors.append("unsupported gate")
        for y in x.get("inputs",[]): check(y)
    for h in m.get("hazards",[]):
        if "logic" not in h: errors.append("hazard without logic")
        else: check(h["logic"])
    return errors
print(validate(model))


In [ ]:
from copy import deepcopy
bad=deepcopy(model)
bad["hazards"][0]["logic"]["inputs"].append({"event":"DOES_NOT_EXIST"})
print(validate(bad))


## 5. Deterministic transformation: hazard logic -> FTA

Mapping rules: an event becomes a BASIC node, a gate becomes an FTA gate, and the hazard ID becomes the top event. This is an explicit algorithm, not an LLM decision.


In [ ]:
def model_to_fta(m,hazard_id):
    errs=validate(m)
    if errs: raise ValueError("; ".join(errs))
    h=next(x for x in m["hazards"] if x["id"]==hazard_id); cat=catalog(m); nodes={}; k=0
    def conv(expr,preferred=None):
        nonlocal k
        if "event" in expr:
            e=expr["event"]; nodes[e]={"type":"BASIC","probability":float(cat[e]["probability"]),"source_component":cat[e]["component"]}; return e
        k+=1; gid=preferred or f"G{k:02d}"; children=[conv(x) for x in expr["inputs"]]
        nodes[gid]={"type":expr["gate"].upper(),"children":children,"source":"hazard_logic"}; return gid
    top=conv(h["logic"],hazard_id)
    return {"top_event":top,"nodes":nodes}
fta=model_to_fta(model,"AUTO_FAIL")
fta


In [ ]:
# The complete TK-101 model also uses basic_events; extend the catalog.
def catalog(m):
    cat = {}
    for component in m.get("components", []):
        for fm in component.get("failure_modes", []):
            cat[fm["id"]] = {**fm, "component": component["id"]}
    for event in m.get("basic_events", []):
        cat[event["id"]] = {**event, "component": "system/basic_event"}
    return cat


## 6. Load the full legacy TK-101 exchange model

`system_model.yaml` stores the original synthetic system description and manually specified hazard logic. Retain it as an independent regression reference for the native SysML import.


In [ ]:
path=ROOT/'data'/'pundit_case'/'system_model.yaml'
with open(path,encoding='utf-8') as f: tk_model=yaml.safe_load(f)
print(tk_model.get('metadata',{})); print('validation:',validate(tk_model))


In [ ]:
tk_fta=model_to_fta(tk_model,'OVERFLOW')
trace=pd.DataFrame([{"fta_node":nid,"type":n['type'],"source_component":n.get('source_component')} for nid,n in tk_fta['nodes'].items()])
trace


## 7. Scope of the legacy route

The YAML example above does not parse SysML and does not discover failure propagation from component connectivity. The engineering annotations and transformation rules determine the output quality. The native adapter below closes the format gap for one explicitly supported SysML modeling convention; it does not eliminate the need for engineering hazard rules.


In [ ]:
from safetycourse.model_io import load_yaml,validate_system_model,generate_fault_tree
lib_model=load_yaml(path); lib_tree=generate_fault_tree(lib_model,'OVERFLOW')
print(validate_system_model(lib_model)); print(lib_tree['top_event'],len(lib_tree['nodes']))


## 8. Native SysML export -> adapter -> FTA

The source is a real SysML v2 textual model with typed parts, probability annotations and Boolean hazard expressions. The reference tool parses and validates it before JSON export. Python then reads that metamodel export, resolves element IDs and transforms supported expressions. No handwritten YAML is passed off as a tool export.

See `docs/sysml_adapter.md` for supported constructs, regeneration and provenance.

In [ ]:
from safetycourse.sysml_adapter import load_sysml_export
from safetycourse.fta import top_event_probability
native_model = load_sysml_export(ROOT/'models/sysml/tk101.export.json')
native_tree = generate_fault_tree(native_model, 'OVERFLOW')
print(native_model['metadata'])
print('Native SysML P(OVERFLOW):', top_event_probability(native_tree))
assert np.isclose(top_event_probability(native_tree), top_event_probability(lib_tree), rtol=1e-12)
pd.DataFrame(native_model['traceability']).head(12)

## Try it!

1. Add a nonexistent event reference and check the validator.
2. Add a component failure mode and use it in a hazard expression.
3. Inspect native source element IDs and generated-node traceability.
4. Change a probability in the `.sysml` model, regenerate its native export and compare the result.
5. Explain why an arbitrary SysML diagram does not contain enough information to infer a correct fault tree.


## Summary

The reproducible chain is **native SysML model -> reference-tool export -> adapter -> validated exchange model -> FTA -> results and traceability**. The Python adapter supports the documented risk-annotated subset, not the complete SysML language. Next, use Bayesian evidence to update probabilistic beliefs.
